# MHD performance

`MHD2D` runs its expensive parts as compiled kernels (Pyccel through cunumpy): the
primitive variables, the MUSCL–HLLD sweeps, the CT electric field, the resistive and
conductive face terms and the stable step. This page shows how to see where the time of
a step goes, with [scope-profiler](https://github.com/max-models/scope-profiler).

carafe's profiling regions carry the tag `carafe` and record nothing until it is turned
on, so they cost nothing in production runs. The regions of a step are

- `carafe.mhd_step` and `carafe.mhd_run`: a whole step and a whole run;
- `carafe.mhd_stable_step`: the CFL and diffusion limit of the step;
- `carafe.mhd_halos`: the halo exchange and the boundary faces;
- `carafe.mhd_fluxes`: the face fluxes and the electric field of one patch.

The tag `cunumpy` adds a region per kernel call (`cunumpy.kernel.<name>`) and `patchwork`
the mesh operations, such as the flux register's update of the cells
(`patchwork.flux_update`).

## Profiling a script

From the command line, turn the tags on for a whole script:

```bash
scope-profiler run --enable-tag carafe --enable-tag cunumpy --enable-tag patchwork -o prof.h5 run.py
scope-profiler inspect prof.h5 --sort total
```

or set `SCOPE_PROFILER_ENABLE_TAGS=carafe,cunumpy,patchwork` for any run. In Python, a
session does the same for a block of code; here the Orszag–Tang vortex on a small grid.
The first step compiles the kernels (once, then they come from a cache), so it runs
before the session.

In [ ]:
import tempfile
from pathlib import Path

import plasma_plots
import scope_profiler as sp

from carafe import MHD2D

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

In [ ]:
vortex = MHD2D.orszag_tang(96)
vortex.step()  # compiles the kernels on the first run
output = Path(tempfile.mkdtemp()) / "mhd.h5"
with sp.session(
    output=output,
    enable_tags=["carafe", "cunumpy", "patchwork"],
    return_results=True,
    verbose=False,
) as run:
    vortex.run(0.05)
steps = vortex.steps - 1
print(f"{steps} steps on {96 * 96} cells")

## The step-time breakdown

`run.results` holds the regions (`scope_profiler.load(output)` reads the file back
later). The exclusive time of a region is its time without the regions inside it; per
step, in milliseconds:

In [ ]:
table = run.results.to_dataframe().set_index("name")
table = table[table.index.str.startswith(("carafe.", "cunumpy.kernel.", "patchwork."))]
per_step = (1e3 * table[["total_duration", "exclusive_duration"]] / steps).rename(
    columns={"total_duration": "total [ms]", "exclusive_duration": "exclusive [ms]"}
)
per_step["calls per step"] = table["num_calls"] / steps
per_step = per_step.drop(index="carafe.mhd_run").sort_values(
    "exclusive [ms]", ascending=False
)
per_step.round(3)

The HLLD sweeps (`cunumpy.kernel.mhd_sweep`, two per patch and stage) take most of
a step; the flux register's update of the cells (`patchwork.flux_update`, one per field
and stage) comes next. The exclusive time of `carafe.mhd_fluxes` and `carafe.mhd_step` is
the Python around the kernels: the start values and the average of the Heun step, and the kernel calls themselves.

In [ ]:
import numpy as np

shares = per_step["exclusive [ms]"]
shares = shares[shares > 0.01 * shares.sum()]  # the regions above 1 %
position = np.arange(shares.size)
with plasma_plots.figure(1, 1) as fig:
    fig[0].bar(position, shares.to_numpy())
    fig[0].set_xticks(position, [name.split(".")[-1] for name in shares.index])
    fig[0].set_ylabel("exclusive time per step [ms]")

## Comparing runs

`scope-profiler diff base.h5 new.h5` compares the regions of two runs, for example before
and after a change; its "Overall" line gives the speedup. A difference of a few percent
needs several runs per side: the times of single runs scatter by about that much.